# DPO Fine-Tuning on Colab

**Method:** Direct Preference Optimization  
**GPU:** T4 (16GB) — Colab Free

Open in Colab: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghassan-gaidi/finetune-lab/blob/main/notebooks/pref/example_dpo_colab.ipynb)

In [ ]:
!pip install -qU transformers accelerate peft bitsandbytes datasets trl huggingface_hub

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import DPOTrainer
from datasets import load_dataset

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # smaller model for free tier

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05)

In [ ]:
dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train[:500]")

def process(row):
    row["chosen"] = tokenizer.apply_chat_template(row["chosen"], tokenize=False)
    row["rejected"] = tokenizer.apply_chat_template(row["rejected"], tokenize=False)
    return row

dataset = dataset.map(process)

In [ ]:
trainer = DPOTrainer(
    model=model,
    ref_model=None,  # TRL creates a reference copy automatically
    args=TrainingArguments(
        output_dir="./dpo-output",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=5e-5,
        bf16=True,
        logging_steps=10,
        save_strategy="epoch",
    ),
    train_dataset=dataset,
    tokenizer=tokenizer,
    max_length=1024,
    max_prompt_length=512,
)

trainer.train()